<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/Makro_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Run command
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""
    <style>
        #scroll_output {
            height: 50px;
            overflow-y: scroll;
            background-color: #1e1e1e;
            color: #00ff00;
            padding: 10px;
            font-family: monospace;
            font-size: 14px;
            border: 1px solid #444;
            display: flex;
            flex-direction: column;
        }
    </style>
    <div id="scroll_output">Starting Makro environment setup...</div>
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            escaped_line = line.replace("'", "\\'").strip()
            display(HTML(f"""<script>
                var obj = document.getElementById('scroll_output');
                obj.innerHTML += '<div>' + '{escaped_line}' + '</div>';
                obj.scrollTop = obj.scrollHeight;
            </script>"""), display_id='scroll_update')
        process.wait()

# 2. Consolidated commands for Makro, Scrapling, and OCR (Updated for Colab Support)
commands_to_run = [
    "apt-get update",
    # Install Chrome Stable, Chromium browser, and Chromedriver dependencies
    "apt-get install -y google-chrome-stable chromium-browser chromium-chromedriver",
    # One big PIP install for all libraries (Scraping + Polars + OCR + Excel + Webdriver Management)
    "pip install selenium bs4 beautifulsoup4 pandas polars xlsxwriter fastexcel curl_cffi scrapling playwright patchright msgspec browserforge nest_asyncio itables chromedriver-autoinstaller webdriver-manager google-colab-selenium easyocr pillow requests -q",
    # Setup the stealth browser
    "patchright install chromium",
    "patchright install-deps"
]

run_and_scroll(commands_to_run)
print("\n✅ Environment ready for Makro! Let's go.")


✅ Environment ready for Makro! Let's go.


### Import Lib

In [ ]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
from google.colab import data_table
data_table.enable_dataframe_formatter()
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-29


In [ ]:
# @title Makro spa clicker
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import re

async def scrape_makro_spa_clicker(start_url: str):
    extracted_data = []
    seen_names = set() # Track names so we don't scrape duplicates if the page is slow

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080}
        )

        page = await context.new_page()
        print("Walking through Makro's front door...")

        # Load the base URL ONCE
        # 1. Change wait_until to 'domcontentloaded' and increase the baseline timeout just in case
        await page.goto(start_url, wait_until="domcontentloaded", timeout=60000)

# 2. Add an explicit wait for a product card to actually appear on the screen.
# (Note: Replace "div.product-card" with the actual CSS class of Makro's product items if it differs)
        try:
            await page.wait_for_selector("div[data-testid='product-card']", timeout=15000)
            # Or whatever the CSS selector for a product card is on Makro
        except Exception as e:
            print(f"Warning: Products took too long to load or page is empty for {start_url}")
        await asyncio.sleep(6) # Wait for initial hydration

        assume_MAX_page = 16

        for page_num in range(1, assume_MAX_page + 1):
            print(f"Scraping page {page_num}...")

            # Scroll to the bottom to force lazy-loaded images and prices to render
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(2)

            # Take a snapshot of the DOM
            html_content = await page.content()
            soup = BeautifulSoup(html_content, "html.parser")

            # Makro wraps product cards in <a> tags linking to /p/ (product pages)
            product_cards = soup.find_all("a", href=lambda href: href and "/p/" in href)

            new_items_found = 0

            for card in product_cards:
                texts = list(card.stripped_strings)
                if len(texts) < 3:
                    continue

                # 1. Hunt for the Product Name
                name = None
                for text in texts:
                    if len(text) > 10 and "฿" not in text and "points" not in text.lower() and "Today" not in text:
                        name = text
                        break

                if not name or name in seen_names:
                    continue

                # 2. Hunt for the Prices (Hyper-Resilient Float Extractor)
                prices = []
                for text in texts:
                    if "buy" in text.lower() or "get" in text.lower() or "point" in text.lower():
                        continue

                    # Strip out currency symbols and commas
                    clean_text = text.replace("฿", "").replace(",", "").strip()

                    try:
                        val = float(clean_text)
                        # Safeguard: Ensure it's an actual price, not a quantity like "3 options"
                        if 5 < val < 10000:
                            prices.append(val)
                    except ValueError:
                        pass

                if not prices:
                    continue

                # 3. The E-Commerce Rule: Smallest is promo, Largest is original
                promo_price = min(prices)
                original_price = max(prices)

                # 4. Hunt for the Discount Condition (e.g., "2 - 2 units")
                condition = None

                # Primary strategy: Target the robust data-test-id you identified
                condition_tag = card.find(lambda tag: tag.has_attr("data-test-id") and "_lbl_buy_more" in tag["data-test-id"])

                if condition_tag:
                    condition = condition_tag.get_text(strip=True)
                else:
                    # Fallback strategy: Read through strings to catch anomalies like "3+ units"
                    for text in texts:
                        if "unit" in text.lower() and any(char.isdigit() for char in text):
                            condition = text
                            break

                # 5. Append everything to the dataset
                extracted_data.append({
                    "name": name,
                    "promotion_price": promo_price,
                    "original_price": original_price,
                    "condition": condition # Pushed the new column here
                })

                seen_names.add(name)
                new_items_found += 1

            print(f"  -> Extracted {new_items_found} new products.")

            # 6. THE SPA CLICKER
            if page_num < assume_MAX_page:
                try:
                    # Find the pagination "Next" button and click it
                    next_button = page.locator("text=Next").first

                    if await next_button.is_visible():
                        await next_button.click()
                        print("  -> Clicked 'Next'. Waiting for SPA to load...")
                        await asyncio.sleep(4) # Give React time to swap the products
                    else:
                        print("  -> 'Next' button not visible. Reached the end of the catalog!")
                        break
                except Exception as e:
                    print(f"  -> Pagination stopped: {e}")
                    break

        await browser.close()

    # --- POLARS CLEANUP ---
    # Explicitly define the schema so Polars doesn't crash if the first 100 items have None as a condition
    df = pl.DataFrame(
        extracted_data,
        schema={
            "name": str,
            "promotion_price": float,
            "original_price": float,
            "condition": str
        }
    )

    if not df.is_empty():
        df = df.unique(subset=["name"], maintain_order=True)

        # Nullify fake promotions
        df = df.with_columns(
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )

        # Calculate the Discount Percentage
        df = df.with_columns(
            pl.when(pl.col("original_price").is_not_null())
            .then(((pl.col("original_price") - pl.col("promotion_price")) / pl.col("original_price") * 100).round(1))
            .otherwise(None)
            .alias("discount_pct")
        ).sort("name")

    return df

In [ ]:
# @title RUN create dataframe

# --- RUN IT ---
# Notice we DO NOT add "&page={}" to the URL anymore. We let the clicker do the work!

scrape_list = [
    "https://www.makro.pro/en/c/collections/Shop%20by%20Brand:%20FINELINE/22199?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJtYXJrZXRpbmdDYXJvdXNlbCUyMiUyQyUyMmJhbm5lck5hbWUlMjIlM0ElMjJTaG9wJTIwYnklMjBCcmFuZCUzQSUyMEZJTkVMSU5FJTIyJTdE",
    "https://www.makro.pro/en/c/household-supplies/laundry-supplies",
    "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20Fresh%20and%20Soft/17985?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTJDJTIyY2Fyb3VzZWxUaXRsZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTdE",
    "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20Dishwash%20Care/17988?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRGlzaHdhc2glMjBDYXJlJTIyJTJDJTIyY2Fyb3VzZWxUaXRsZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRGlzaHdhc2glMjBDYXJlJTIyJTdE"
]

scraped_dfs = []

for url in scrape_list:
    print(f"\nStarting scrape for URL: {url}")
    df_result = await scrape_makro_spa_clicker(start_url=url)

    print("=" * 60)
    print(f"Extracted {len(df_result)} products from this category.")
    print("=" * 60)

    # Only append if the dataframe actually caught data to avoid schema/concat errors
    if not df_result.is_empty():
        scraped_dfs.append(df_result)

# Concatenate all collected DataFrames into one
if scraped_dfs:
    df_combined_makro = pl.concat(scraped_dfs)

    # Optional: Drop any duplicate products in case categories overlap
    df_combined_makro = df_combined_makro.unique(subset=["name"], maintain_order=True)
else:
    # Fallback just in case everything times out
    df_combined_makro = pl.DataFrame()

print(f"\nMakro Scraping Fully Complete - {len(df_combined_makro)} total unique products.")
print(df_combined_makro)


Starting scrape for URL: https://www.makro.pro/en/c/collections/Shop%20by%20Brand:%20FINELINE/22199?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJtYXJrZXRpbmdDYXJvdXNlbCUyMiUyQyUyMmJhbm5lck5hbWUlMjIlM0ElMjJTaG9wJTIwYnklMjBCcmFuZCUzQSUyMEZJTkVMSU5FJTIyJTdE
Walking through Makro's front door...
Scraping page 1...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 2...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 3...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 4...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 5...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 6...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 7...
  -> Extracted 6 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 8...


## Try condition

In [ ]:
# @title Load watchlists with "condition"
import json
import time
import re
import polars as pl
from bs4 import BeautifulSoup
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. Setup Headless Chrome
driver = gs.Chrome()

# 2. Watchlist
urls_watchlist = [
# Fineline Liquid Detergent Plus Sunny Gold 550 ML. Gold
"https://www.makro.pro/en/p/BxZe5f5-174668393051076",
# Fineline Liquid Detergent Plus Sunny Gold 1250 ML.
"https://www.makro.pro/en/p/z0icJlu-272067985587301",
# PAO Win Wash Concentrated Liquid Detergent Orange 620 ml
"https://www.makro.pro/en/p/ex0iyeb-7078772015299",
# PAO Win Wash Concentrated Liquid Detergent Orange 1.3 l
"https://www.makro.pro/en/p/gka-ofe-7352769970371",
# PAO Super Soft Laundry Detergent 1.8 kg x 1+1
"https://www.makro.pro/en/p/piHDQ_qh-227017132155382",
# PAO Super White Standard Formula Powder Detergent 2.4 kg
"https://www.makro.pro/en/p/ru-ickh-6761207398595",
# ATTACK Easy Regular Detergent Happy Sweet Pink 2.3/2.5 kg
"https://www.makro.pro/en/p/vzcfnxp-7078770016451",
# ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1
"https://www.makro.pro/en/p/4_Yu5hcU-829783173153864",
# PRO Regular Powder Detergent Blue Plus Red 1.7 kg x 1+1
"https://www.makro.pro/en/p/05CJD5Oy-126255086281148",
# PRO Regular Powder Detergent Blue Plus Red 2.4 kg
"https://www.makro.pro/en/p/hojuwx2-6761206448323",
# HYGIENE Fabric Softener Expert Care Milky Touch White 480 ml
"https://www.makro.pro/en/p/iuia8gd-7416044290243",
# HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS
"https://www.makro.pro/en/p/ampyy_n-7275653693635",
# HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l
"https://www.makro.pro/en/p/854707-7248018604227",
# HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1
"https://www.makro.pro/en/p/8vSXFyA-869126234793421",
# LIPON F Dishwash 500 ml x 3
"https://www.makro.pro/en/p/8lbyz1p-6761190785219",
# LIPON F DISHWASHING LIQUID 3.2 L.
"https://www.makro.pro/en/p/bpm7o_y-6761191080131",
]

scraped_data = []

# Strict regex to only capture actual prices (e.g., "฿ 119", "฿ 1,200.00")
price_pattern = re.compile(r'^\s*฿\s*[\d,]+(\.\d+)?\s*$')

# 3. Scraping Loop
for url in urls_watchlist:
    print(f"Scraping: {url}")
    driver.get(url)

    # --- SMART WAIT ---
    # Wait up to 10 seconds for the product title element to appear in the DOM
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//*[contains(@data-test-id, '_product_title')]"))
        )
        # Give the JS a tiny 1-second buffer to finish injecting the pricing text
        time.sleep(1)
    except Exception as e:
        print(f"Page load timed out for {url}")

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # --- EXTRACT NAME ---
    name_div = soup.find('div', attrs={'data-test-id': lambda x: x and x.endswith('_product_title')})
    product_name = name_div.text.strip() if name_div else "Name not found"

    # --- EXTRACT CONDITIONS (Tiers) ---
    tier_divs = soup.find_all('div', attrs={'data-test-id': lambda x: x and x.startswith('unit_tier_')})
    conditions = [tier.text.strip() for tier in tier_divs]

    condition_dict = {}

    # --- EXTRACT PRICES (Only if conditions exist) ---
    if conditions:
        # Find all valid price spans on the page
        price_tags = soup.find_all(lambda tag: tag.name == 'span' and tag.text and price_pattern.match(tag.text.strip()))
        all_valid_prices = [tag.text.strip() for tag in price_tags]

        # Match the exact number of tiers to the bottom-most prices
        prices = all_valid_prices[-len(conditions):]

        condition_dict = dict(zip(conditions, prices))

    # Store as JSON string
    condition_json = json.dumps(condition_dict, ensure_ascii=False)

    scraped_data.append({
        "url": url,
        "name": product_name,
        "condition": condition_json
    })

driver.quit()

# 4. Show DataFrame
print("\n--- Final Polars DataFrame ---")
df_watchlist_cond = pl.DataFrame(scraped_data)
pl.Config.set_fmt_str_lengths(100)
# replace "{}" with Null
df_watchlist_cond = df_watchlist_cond.with_columns(
    pl.col("condition").replace("{}", None)
)
print(df_watchlist_cond)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Scraping: https://www.makro.pro/en/p/BxZe5f5-174668393051076
Scraping: https://www.makro.pro/en/p/z0icJlu-272067985587301
Scraping: https://www.makro.pro/en/p/ex0iyeb-7078772015299
Scraping: https://www.makro.pro/en/p/gka-ofe-7352769970371
Scraping: https://www.makro.pro/en/p/piHDQ_qh-227017132155382
Scraping: https://www.makro.pro/en/p/ru-ickh-6761207398595
Scraping: https://www.makro.pro/en/p/vzcfnxp-7078770016451
Scraping: https://www.makro.pro/en/p/4_Yu5hcU-829783173153864
Scraping: https://www.makro.pro/en/p/05CJD5Oy-126255086281148
Scraping: https://www.makro.pro/en/p/hojuwx2-6761206448323
Scraping: https://www.makro.pro/en/p/iuia8gd-7416044290243
Scraping: https://www.makro.pro/en/p/ampyy_n-7275653693635
Scraping: https://www.makro.pro/en/p/854707-7248018604227
Scraping: https://www.makro.pro/en/p/8vSXFyA-869126234793421
Scraping: https://www.makro.pro/en/p/8lbyz1p-6761190785219
Scraping: https://www.makro.pro/en/p/bpm7o_y-6761191080131

--- Final Polars DataFrame ---
shape: (16

In [ ]:
df_watchlist_cond.to_pandas()

,url,name,condition
0,https://www.makro.pro/en/p/BxZe5f5-17466839305...,Fineline Liquid Detergent Plus Sunny Gold 550 ...,None
1,https://www.makro.pro/en/p/z0icJlu-27206798558...,Fineline Liquid Detergent Plus Sunny Gold 1250...,"{""1 - 1 units"": ""฿ 110"", ""2+ units"": ""฿ 98""}"
2,https://www.makro.pro/en/p/ex0iyeb-7078772015299,PAO Win Wash Concentrated Liquid Detergent Ora...,None
3,https://www.makro.pro/en/p/gka-ofe-7352769970371,PAO Win Wash Concentrated Liquid Detergent Ora...,None
4,https://www.makro.pro/en/p/piHDQ_qh-2270171321...,PAO Super Soft Laundry Detergent 1.8 kg x 1+1,None
5,https://www.makro.pro/en/p/ru-ickh-6761207398595,PAO Super White Standard Formula Powder Deterg...,None
6,https://www.makro.pro/en/p/vzcfnxp-7078770016451,ATTACK Easy Regular Detergent Happy Sweet Pink...,None
7,https://www.makro.pro/en/p/4_Yu5hcU-8297831731...,ATTACK Easy Conventional Detergent Happy Sweet...,None
8,https://www.makro.pro/en/p/05CJD5Oy-1262550862...,PRO Regular Powder Detergent Blue Plus Red 1.7...,None
9,https://www.makro.pro/en/p/hojuwx2-6761206448323,PRO Regular Powder Detergent Blue Plus Red 2.4 kg,None


In [ ]:
# @title watchlist STREAMLINED
import asyncio
import re
import polars as pl
from playwright.async_api import async_playwright

async def scrape_makro_product(url, browser_instance, semaphore):
    async with semaphore:
        context = await browser_instance.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080}
        )
        page = await context.new_page()

        # Pre-fill the dictionary with None in case of a crash
        data = {
            "url": url,
            "name": None,
            "promotion_price": None,
            "original_price": None
        }

        try:
            print(f"Scraping: {url}")
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            title_selector = '[data-test-id$="_product_title"], h1'
            try:
                await page.wait_for_selector(title_selector, state="visible", timeout=20000)
            except Exception:
                print(f"  -> Warning: Title selector timed out for {url}. Attempting to parse anyway...")

            await page.wait_for_timeout(1500)

            # Extract the title and RAW visible text
            page_data = await page.evaluate('''() => {
                let titleEl = document.querySelector('[data-test-id$="_product_title"]');
                if (!titleEl) {
                    titleEl = document.querySelector('h1');
                }
                if (!titleEl) return null;

                return {
                    name: titleEl.innerText.trim(),
                    rawText: document.body.innerText
                };
            }''')

            promo_price = None
            original_price = None

            # Python-side Text Parsing
            if page_data and page_data['rawText']:
                lines = [line.strip() for line in page_data['rawText'].split('\n') if line.strip()]

                start_idx = -1
                for i, line in enumerate(lines):
                    if "Code :" in line or "Code:" in line:
                        start_idx = i
                        break

                if start_idx == -1 and page_data['name']:
                    for i, line in enumerate(lines):
                        if page_data['name'] in line:
                            start_idx = i

                if start_idx != -1:
                    prices = []
                    for line in lines[start_idx+1 : start_idx+16]:
                        clean_line = line.replace("฿", "").replace(",", "").strip()
                        if re.match(r'^\d+(\.\d+)?$', clean_line):
                            prices.append(clean_line)

                    if len(prices) >= 2:
                        promo_price = prices[0]
                        original_price = prices[1]
                    elif len(prices) == 1:
                        promo_price = prices[0]
                        original_price = prices[0]

            # Update the dictionary if parsing succeeded
            data["name"] = page_data['name'] if page_data else None
            data["promotion_price"] = promo_price
            data["original_price"] = original_price

        except Exception as e:
            print(f"  -> Failed to scrape {url} entirely: {e}")
            # The data dictionary will just return None for the name and prices
        finally:
            await context.close()

        return data

async def main(urls):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        semaphore = asyncio.Semaphore(3)

        tasks = [scrape_makro_product(url, browser, semaphore) for url in urls]
        results = await asyncio.gather(*tasks)

        await browser.close()
        return results

# --- Execution ---
urls_watchlist = [
    # Fineline Liquid Detergent Plus Sunny Gold 550 ML. Gold
    "https://www.makro.pro/en/p/BxZe5f5-174668393051076",
    # Fineline Liquid Detergent Plus Sunny Gold 1250 ML.
    "https://www.makro.pro/en/p/z0icJlu-272067985587301",
    # PAO Win Wash Concentrated Liquid Detergent Orange 620 ml
    "https://www.makro.pro/en/p/ex0iyeb-7078772015299",
    # PAO Win Wash Concentrated Liquid Detergent Orange 1.3 l
    "https://www.makro.pro/en/p/gka-ofe-7352769970371",
    # PAO Super Soft Laundry Detergent 1.8 kg x 1+1
    "https://www.makro.pro/en/p/piHDQ_qh-227017132155382",
    # PAO Super White Standard Formula Powder Detergent 2.4 kg
    "https://www.makro.pro/en/p/ru-ickh-6761207398595",
    # ATTACK Easy Regular Detergent Happy Sweet Pink 2.3/2.5 kg
    "https://www.makro.pro/en/p/vzcfnxp-7078770016451",
    # ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1
    "https://www.makro.pro/en/p/4_Yu5hcU-829783173153864",
    # PRO Regular Powder Detergent Blue Plus Red 1.7 kg x 1+1
    "https://www.makro.pro/en/p/05CJD5Oy-126255086281148",
    # PRO Regular Powder Detergent Blue Plus Red 2.4 kg
    "https://www.makro.pro/en/p/hojuwx2-6761206448323",
    # HYGIENE Fabric Softener Expert Care Milky Touch White 480 ml
    "https://www.makro.pro/en/p/iuia8gd-7416044290243",
    # HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS
    "https://www.makro.pro/en/p/ampyy_n-7275653693635",
    # HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l
    "https://www.makro.pro/en/p/854707-7248018604227",
    # HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1
    "https://www.makro.pro/en/p/8vSXFyA-869126234793421",
    # LIPON F Dishwash 500 ml x 3
    "https://www.makro.pro/en/p/8lbyz1p-6761190785219",
    # LIPON F DISHWASHING LIQUID 3.2 L.
    "https://www.makro.pro/en/p/bpm7o_y-6761191080131",
]

raw_data = await main(urls_watchlist)

# Explicitly define schema to prevent typing errors if URLs crash or prices are None
df_watchlist_price = pl.DataFrame(
    raw_data,
    schema={
        "url": str,
        "name": str,
        "promotion_price": str,
        "original_price": str
    }
)

print("\n--- Final Polars DataFrame ---")
print(df_watchlist_price)

Scraping: https://www.makro.pro/en/p/BxZe5f5-174668393051076
Scraping: https://www.makro.pro/en/p/z0icJlu-272067985587301
Scraping: https://www.makro.pro/en/p/ex0iyeb-7078772015299
Scraping: https://www.makro.pro/en/p/gka-ofe-7352769970371
Scraping: https://www.makro.pro/en/p/piHDQ_qh-227017132155382
Scraping: https://www.makro.pro/en/p/ru-ickh-6761207398595
Scraping: https://www.makro.pro/en/p/vzcfnxp-7078770016451
Scraping: https://www.makro.pro/en/p/4_Yu5hcU-829783173153864
Scraping: https://www.makro.pro/en/p/05CJD5Oy-126255086281148
Scraping: https://www.makro.pro/en/p/hojuwx2-6761206448323
Scraping: https://www.makro.pro/en/p/iuia8gd-7416044290243
Scraping: https://www.makro.pro/en/p/ampyy_n-7275653693635
Scraping: https://www.makro.pro/en/p/854707-7248018604227
Scraping: https://www.makro.pro/en/p/8vSXFyA-869126234793421
Scraping: https://www.makro.pro/en/p/8lbyz1p-6761190785219
Scraping: https://www.makro.pro/en/p/bpm7o_y-6761191080131

--- Final Polars DataFrame ---
shape: (16

In [ ]:
df_merge_watchlist = df_watchlist_price.join(df_watchlist_cond, on="url", how="left")
df_merge_watchlist

url,name,promotion_price,original_price,name_right,condition
str,str,str,str,str,str
"""https://www.makro.pro/en/p/BxZe5f5-174668393051076""","""Fineline Liquid Detergent Plus Sunny Gold 550 ML. Gold""","""66""","""89.00""","""Fineline Liquid Detergent Plus Sunny Gold 550 ML. Gold""",null
"""https://www.makro.pro/en/p/z0icJlu-272067985587301""","""Fineline Liquid Detergent Plus Sunny Gold 1250 ML.""","""98""","""179.00""","""Fineline Liquid Detergent Plus Sunny Gold 1250 ML.""","""{""1 - 1 units"": ""฿ 110"", ""2+ units"": ""฿ 98""}"""
"""https://www.makro.pro/en/p/ex0iyeb-7078772015299""","""PAO Win Wash Concentrated Liquid Detergent Orange 620 ml""","""89""","""89""","""PAO Win Wash Concentrated Liquid Detergent Orange 620 ml""",null
"""https://www.makro.pro/en/p/gka-ofe-7352769970371""","""PAO Win Wash Concentrated Liquid Detergent Orange 1.3 l""","""165""","""165""","""PAO Win Wash Concentrated Liquid Detergent Orange 1.3 l""",null
"""https://www.makro.pro/en/p/piHDQ_qh-227017132155382""","""PAO Super Soft Laundry Detergent 1.8 kg x 1+1""","""149""","""164.00""","""PAO Super Soft Laundry Detergent 1.8 kg x 1+1""",null
…,…,…,…,…,…
"""https://www.makro.pro/en/p/ampyy_n-7275653693635""","""HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS""","""117""","""117""","""HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS""",null
"""https://www.makro.pro/en/p/854707-7248018604227""","""HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l""","""130""","""130""","""HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l""",null
"""https://www.makro.pro/en/p/8vSXFyA-869126234793421""","""HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1""","""189""","""209.00""","""HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1""",null


In [ ]:
# Function to ensure consistent schema for relevant columns
def ensure_consistent_schema(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("name").cast(pl.String, strict=False).alias("name"),
        pl.col("promotion_price").cast(pl.Float64, strict=False).alias("promotion_price"),
        pl.col("original_price").cast(pl.Float64, strict=False).alias("original_price"),
        pl.col("condition").cast(pl.String, strict=False).alias("condition"),
    )
    return_cols = ["name", "promotion_price", "original_price", "condition"]
    return df.select(return_cols)

# Apply the schema fix to each DataFrame before concatenation
df_combined_makro = ensure_consistent_schema(df_combined_makro)
df_watchlist_fixed = ensure_consistent_schema(df_merge_watchlist)

df_makro = pl.concat([
    df_combined_makro,
    df_watchlist_fixed
])

## Transform Makro
if "discount_pct" in df_makro.columns:
    df_makro = df_makro.drop("discount_pct")

In [ ]:
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# How to use it:
df_prep_makro = re_evaluate_price(df_makro)

In [ ]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int64, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [ ]:
df_trans_makro = parse_product_names(df_prep_makro, "Makro")

In [ ]:
df_trans_makro

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""FINELINE CONCENTRATED FABRIC SOFTENER ALL DAY FRESH Blooming Fresh scent 1150 ML. Pink""",113.0,169.0,"""2+ units""","""2026-04-29""","""FINELINE""",1150,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC SOFTENER ALL DAY FRESH Blooming Fresh scent 450 ML. Pink""",51.0,69.0,null,"""2026-04-29""","""FINELINE""",450,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC SOFTENER ALL DAY FRESH Morning Fresh scent 1150 ML. blue""",113.0,169.0,"""2+ units""","""2026-04-29""","""FINELINE""",1150,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC SOFTENER ALL DAY FRESH Morning Fresh scent 450 ML. blue""",53.0,69.0,"""2 - 2 units""","""2026-04-29""","""FINELINE""",450,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC SOFTENER HAPPINESS PEACH BLOSSOM 1150 ML.""",107.0,169.0,"""2 - 2 units""","""2026-04-29""","""FINELINE""",1150,"""ML""",null,"""Makro"""
…,…,…,…,…,…,…,…,…,…
"""HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS""",null,117.0,null,"""2026-04-29""","""HYGIENE""",480,"""ML""","""X 2""","""Makro"""
"""HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l""",null,130.0,null,"""2026-04-29""","""HYGIENE""",1,"""L""",null,"""Makro"""
"""HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1""",189.0,209.0,null,"""2026-04-29""","""HYGIENE""",1,"""L""","""X 1""","""Makro"""


In [ ]:
df_trans_makro.write_excel(f"makro_{today_date}.xlsx")

### Search watchlist

In [ ]:
list_to_search = [
# --- Makro ---
"Fineline Liquid Detergent Plus Sunny Gold 550 ML. Gold",
"Fineline Liquid Detergent Plus Sunny Gold 1250 ML.",
"HYGIENE Expert Wash Liauid Detergent Milky Touch 1.4 ml",
"PAO Win Wash Concentrated Liquid Detergent Orange 620 ml",
"PAO Win Wash Concentrated Liquid Detergent Orange 1.3 l",
"PAO Super White Laundry Detergent 1.8 kg x 1+1",
"PAO Super White Standard Formula Powder Detergent 2.4 kg",
"ATTACK Easy Regular Detergent Happy Sweet Pink 2.3/2.5 kg",
"ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1",
"PRO Regular Powder Detergent Blue Plus Red 1.7 kg x 1+1",
"PRO Regular Powder Detergent Blue Plus Red 2.4 kg",
"HYGIENE Fabric Softener Expert Care Milky Touch White 480 ml",
"HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS",
"HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l",
"HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1",
"LIPON F Dishwash 500 ml x 3",
"LIPON F DISHWASHING LIQUID 3.2 L.",
]

In [ ]:
search_df = pl.DataFrame({"product_name": list_to_search})

# Join with df_trans_makro to get original_price and promotion_price
search_results_df = search_df.join(
    df_trans_makro.select(["name", "original_price", "promotion_price"]),
    left_on="product_name",
    right_on="name",
    how="left"
)
makro_names_set = set(df_trans_makro["name"].to_list())

search_results_df = search_results_df.with_columns(
    pl.col("product_name").is_in(makro_names_set).alias("Found")
).unique()

print("Search Results with Prices:")
print(search_results_df)

Search Results with Prices:
shape: (21, 4)
┌─────────────────────────────────┬────────────────┬─────────────────┬───────┐
│ product_name                    ┆ original_price ┆ promotion_price ┆ Found │
│ ---                             ┆ ---            ┆ ---             ┆ ---   │
│ str                             ┆ f64            ┆ f64             ┆ bool  │
╞═════════════════════════════════╪════════════════╪═════════════════╪═══════╡
│ Fineline Liquid Detergent Plus… ┆ 89.0           ┆ 66.0            ┆ true  │
│ Fineline Liquid Detergent Plus… ┆ 89.0           ┆ 66.0            ┆ true  │
│ Fineline Liquid Detergent Plus… ┆ 179.0          ┆ 98.0            ┆ true  │
│ HYGIENE Expert Wash Liauid Det… ┆ 135.0          ┆ 105.0           ┆ true  │
│ PAO Win Wash Concentrated Liqu… ┆ 89.0           ┆ null            ┆ true  │
│ …                               ┆ …              ┆ …               ┆ …     │
│ HYGIENE Expert Care Concentrat… ┆ 209.0          ┆ 189.0           ┆ true  │
│ LIPON F

In [ ]:
search_results_df.to_pandas()

,product_name,original_price,promotion_price,Found
0,Fineline Liquid Detergent Plus Sunny Gold 550 ...,89.0,66.0,True
1,Fineline Liquid Detergent Plus Sunny Gold 550 ...,89.0,66.0,True
2,Fineline Liquid Detergent Plus Sunny Gold 1250...,179.0,98.0,True
3,HYGIENE Expert Wash Liauid Detergent Milky Tou...,135.0,105.0,True
4,PAO Win Wash Concentrated Liquid Detergent Ora...,89.0,NaN,True
5,PAO Win Wash Concentrated Liquid Detergent Ora...,165.0,82.5,True
6,PAO Super White Laundry Detergent 1.8 kg x 1+1,179.0,155.0,True
7,PAO Super White Standard Formula Powder Deterg...,155.0,119.0,True
8,PAO Super White Standard Formula Powder Deterg...,155.0,119.0,True
9,ATTACK Easy Regular Detergent Happy Sweet Pink...,159.0,125.0,True


In [ ]:
[
# Fineline Liquid Detergent Plus Sunny Gold 550 ML. Gold
"https://www.makro.pro/en/p/BxZe5f5-174668393051076",
# Fineline Liquid Detergent Plus Sunny Gold 1250 ML.
"https://www.makro.pro/en/p/z0icJlu-272067985587301",
# PAO Win Wash Concentrated Liquid Detergent Orange 620 ml
"https://www.makro.pro/en/p/ex0iyeb-7078772015299",
# PAO Win Wash Concentrated Liquid Detergent Orange 1.3 l
"https://www.makro.pro/en/p/gka-ofe-7352769970371",
# PAO Super Soft Laundry Detergent 1.8 kg x 1+1
"https://www.makro.pro/en/p/piHDQ_qh-227017132155382",
# PAO Super White Standard Formula Powder Detergent 2.4 kg
"https://www.makro.pro/en/p/ru-ickh-6761207398595",
# ATTACK Easy Regular Detergent Happy Sweet Pink 2.3/2.5 kg
"https://www.makro.pro/en/p/vzcfnxp-7078770016451",
# ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1
"https://www.makro.pro/en/p/4_Yu5hcU-829783173153864",
# PRO Regular Powder Detergent Blue Plus Red 1.7 kg x 1+1
"https://www.makro.pro/en/p/05CJD5Oy-126255086281148",
# PRO Regular Powder Detergent Blue Plus Red 2.4 kg
"https://www.makro.pro/en/p/hojuwx2-6761206448323",
# HYGIENE Fabric Softener Expert Care Milky Touch White 480 ml
"https://www.makro.pro/en/p/iuia8gd-7416044290243",
# HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS
"https://www.makro.pro/en/p/ampyy_n-7275653693635",
# HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l
"https://www.makro.pro/en/p/854707-7248018604227",
# HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1
"https://www.makro.pro/en/p/8vSXFyA-869126234793421"
# LIPON F Dishwash 500 ml x 3
"https://www.makro.pro/en/p/8lbyz1p-6761190785219"
# LIPON F DISHWASHING LIQUID 3.2 L.
"https://www.makro.pro/en/p/bpm7o_y-6761191080131"
]

'https://www.makro.pro/en/p/8vSXFyA-869126234793421'